In [ ]:
import json
import random

random.seed(42)

# read jsonl and write as csv with header
train = open('../datasets/amazon_digital_music_train.csv', 'w')
valdn = open('../datasets/amazon_digital_music_valdn.csv', 'w')
train.write('rating#text#title\n')
valdn.write('rating#text#title\n')

hash_in_text = 0
text_lengths = []
title_lengths = []
logest_text_line = ''
logest_title_line = ''

too_long = 0

with open('../datasets/amazon_digital_music.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        if data['verified_purchase']:
            text = data['text']
            title = data['title']
            
            if '#' in text or '#' in title:
                hash_in_text += 1
                continue
            
            text_length = len(text.split())
            title_length = len(title.split())
            text_lengths.append(text_length)
            title_lengths.append(title_length)
            if text_length > 128 or title_length > 32:
                too_long += 1
                continue
            
            if text_length == 0 or title_length == 0:
                print("text: ", text, "| title:", title)
                print('-'*100)
                continue

            if random.random() <= 0.9:
                train.write(f"{data['rating']}#{text}#{title}\n")
            else:
                valdn.write(f"{data['rating']}#{text}#{title}\n")

print(f"Number of reviews with hash in text: {hash_in_text}", f"Longest text: {max(text_lengths)}", f"Longest title: {max(title_lengths)}", f"Too long: {too_long}")

train.close()
valdn.close()

In [ ]:
# read google word2vec model

import gensim
# read file from https://groups.google.com/g/word2vec-toolkit
model = gensim.models.KeyedVectors.load_word2vec_format('../datasets/GoogleNews-vectors-negative300.bin', binary=True)

# print the first 10 words
print(model.index_to_key[:10])

# print the first 10 vectors
print(model.get_vector('the')[:10])

In [ ]:
# create a word2vec file from model, wheere we write a word (as text), followed by a zero byte, followed by 300 raw floats
count = 0
raw_file = open("../datasets/google_word2vec.bin", "wb")
for i, word in enumerate(model.index_to_key):
    utf8_word = word.encode('utf-8')
    raw_file.write(utf8_word)
    raw_file.write(b'\r')
    vec = model.get_vector(word)
    raw_file.write(vec.tobytes())
print("number of words: ", i)
raw_file.close()




In [ ]:
import pandas as pd
import numpy as np

# from https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition
df = pd.read_csv('../datasets/ObesityDataSet_raw_and_data_sinthetic.csv')

# replace all non-numeric values with unique integers:
non_numeric_cols = df.select_dtypes(include=['object']).columns

# replace all non-numeric values with unique integers:
for col in non_numeric_cols:
    df[col] = df[col].astype('category').cat.codes

# shuffle the dataframe, and split into train and valdn
df = df.sample(frac=1).reset_index(drop=True)
train_df = df.iloc[:int(0.8*len(df))]
val_df = df.iloc[int(0.8*len(df)):]


# write the train and valdn to csv
train_df.to_csv('../datasets/obesity_train.csv', index=False)
val_df.to_csv('../datasets/obesity_valdn.csv', index=False)